# Dead-Zone Prediction Model v3 — Training Notebook

**Project:** Network Cell Analyzer (EECE 451)  
**Model:** Dual LightGBM (Regressor + Stacked Classifier)  
**Region:** Lebanon  

**Runtime on Colab A100: ~8-12 minutes total** (mostly data download + Ookla filtering)

---

## Design Philosophy

### Labels come from REAL measurements only

| Tier | Source | Measurement type | Weight |
|------|--------|-----------------|--------|
| **1** | App signal readings | RSRP from device radio (3GPP TS 36.133) | 3.0 |
| **2** | Ookla speed tests | Real throughput from real users (ITU-T Y.1541) | 2.0 |
| **3** | Tower topology | COST-231 physics estimate (gap-filler only) | 0.5 |

**Rule:** OpenCelliD towers and physics models are *features*, never label sources.


---
## 0. Environment Setup
This cell installs packages, clones the project repo, and downloads all datasets.
**Just press Run All — everything is automatic.**


In [1]:
# ──────────────────────────────────────────────────────
# 0a. Install dependencies
# ──────────────────────────────────────────────────────
!pip install -q lightgbm h3 shap seaborn pyarrow

import sys, os, json, gzip, warnings, sqlite3, time, subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import joblib
import lightgbm as lgb

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
sns.set_style('whitegrid')

print(f'NumPy    : {np.__version__}')
print(f'Pandas   : {pd.__version__}')
print(f'LightGBM : {lgb.__version__}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.1 MB/s eta 0:00:00
NumPy    : 2.0.2
Pandas   : 2.2.2
LightGBM : 4.6.0


In [2]:
# ──────────────────────────────────────────────────────
# 0b. Mount Google Drive for saving results
# ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = Path('/content/drive/MyDrive/EECE451_DeadZone')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print(f'Drive output: {DRIVE_OUT}')

Mounted at /content/drive
Drive output: /content/drive/MyDrive/EECE451_DeadZone


In [3]:
# ──────────────────────────────────────────────────────
# 0c. Clone project repo (public GitHub)
# ──────────────────────────────────────────────────────
REPO_URL = 'https://github.com/rae81/EECE451-Project.git'
REPO_DIR = Path('/content/EECE451-Project')

if not REPO_DIR.exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already cloned at {REPO_DIR}')

# Set project paths
SERVER_DIR = REPO_DIR / 'server'
DATA_DIR = SERVER_DIR / 'data' / 'raw'
OUTPUT_DIR = SERVER_DIR / 'instance' / 'ml'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'reports').mkdir(exist_ok=True)

# Add server/ to Python path so project modules are importable
if str(SERVER_DIR) not in sys.path:
    sys.path.insert(0, str(SERVER_DIR))

os.chdir(str(SERVER_DIR))
print(f'Working dir: {Path.cwd()}')
print(f'Data dir   : {DATA_DIR}')
print(f'Output dir : {OUTPUT_DIR}')


Cloning into '/content/EECE451-Project'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 184 (delta 45), reused 118 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 271.12 KiB | 4.17 MiB/s, done.
Resolving deltas: 100% (45/45), done.
Working dir: /content/EECE451-Project/server
Data dir   : /content/EECE451-Project/server/data/raw
Output dir : /content/EECE451-Project/server/instance/ml


In [4]:
# ──────────────────────────────────────────────────────
# 0d. Download datasets from public sources
# ──────────────────────────────────────────────────────
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_DIR.mkdir(parents=True, exist_ok=True)
BBOX = (33.05, 35.09, 34.72, 36.68)  # Lebanon

# Overpass mirrors — automatic failover
OVERPASS_ENDPOINTS = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://maps.mail.ru/osm/tools/overpass/api/interpreter',
]

# Retry-capable session
session = requests.Session()
session.mount('https://', HTTPAdapter(
    max_retries=Retry(total=3, backoff_factor=2,
                      status_forcelist=[429, 500, 502, 503, 504])))

def download_file(url, dest, desc=''):
    """Download a file if it doesn't already exist."""
    if dest.exists() and dest.stat().st_size > 100:
        print(f'  [skip] {desc or dest.name} already exists ({dest.stat().st_size/1e6:.1f} MB)')
        return
    print(f'  [downloading] {desc or dest.name} ...')
    resp = session.get(url, stream=True, timeout=600)
    resp.raise_for_status()
    with open(dest, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print(f'    Done: {dest.stat().st_size/1e6:.1f} MB')

def overpass_query(query, dest, desc=''):
    """Run Overpass query with mirror failover and retry."""
    if dest.exists() and dest.stat().st_size > 100:
        print(f'  [skip] {desc or dest.name} already exists')
        return
    print(f'  [querying] {desc or dest.name} ...')
    last_err = None
    for endpoint in OVERPASS_ENDPOINTS:
        try:
            resp = session.post(endpoint, data={'data': query}, timeout=360)
            resp.raise_for_status()
            data = resp.json()
            with open(dest, 'w') as f:
                json.dump(data, f)
            n = len(data.get('elements', []))
            print(f'    Done: {n:,} elements ({dest.stat().st_size/1e6:.1f} MB)')
            return
        except Exception as e:
            last_err = e
            print(f'    [warn] {endpoint.split("//")[1].split("/")[0]} failed: {e}')
            time.sleep(5)
    print(f'    [FAILED] All mirrors failed for {desc}. Will use empty DataFrame.')

print('=== Downloading datasets ===')

# 1. Ookla Open Data (publicly accessible S3 bucket)
OOKLA_URL = (
    'https://ookla-open-data.s3.amazonaws.com/parquet/performance/'
    'type=mobile/year=2024/quarter=4/'
    '2024-10-01_performance_mobile_tiles.parquet'
)
ookla_path = DATA_DIR / 'ookla_mobile_q4_2025.parquet'
download_file(OOKLA_URL, ookla_path, 'Ookla mobile Q4 2024 (~170 MB)')

# 2. OSM telecom infrastructure (small query ~200 nodes)
overpass_query(
    '[out:json][timeout:120][bbox:33.05,35.09,34.72,36.68];'
    '(node["man_made"~"tower|mast"]["tower:type"~"communication|telecom"];'
    'node["telecom"];way["man_made"~"tower|mast"]["tower:type"~"communication|telecom"];);'
    'out center qt;',
    DATA_DIR / 'osm_telecom_lebanon.json',
    'OSM telecom sites'
)

# 3. OSM buildings — sample only (full Lebanon is too large for Overpass)
#    Use major city bboxes to keep query under Overpass limits
overpass_query(
    '[out:json][timeout:180];'
    '(way["building"](33.82,35.42,33.95,35.58);'  # Beirut metro
    'way["building"](34.40,35.80,34.48,35.90);'    # Tripoli
    'way["building"](33.82,35.47,33.90,35.55);'    # Jounieh
    'way["building"](33.26,35.18,33.30,35.22);'    # Tyre
    'way["building"](33.55,35.35,33.60,35.40);'    # Sidon
    ');out center qt 30000;',
    DATA_DIR / 'osm_buildings_lebanon.json',
    'OSM buildings (major cities)'
)

# 4. OSM roads — limit to main road types
overpass_query(
    '[out:json][timeout:180][bbox:33.05,35.09,34.72,36.68];'
    'way["highway"~"motorway|trunk|primary|secondary|tertiary"];'
    'out center qt 20000;',
    DATA_DIR / 'osm_roads_lebanon.json',
    'OSM roads'
)

# 5. Coastline (small query)
overpass_query(
    '[out:json][timeout:120][bbox:33.05,35.09,34.72,36.68];'
    'way["natural"="coastline"];out geom qt;',
    DATA_DIR / 'lebanon_coastline.json',
    'Coastline'
)

# 6. DEM — OpenTopoData API (rate-limited, batched)
dem_path = DATA_DIR / 'lebanon_dem_dense.csv'
if not dem_path.exists() or dem_path.stat().st_size < 100:
    print('  [computing] DEM grid via OpenTopoData API ...')
    lats = np.arange(33.05, 34.72, 0.02)
    lons = np.arange(35.09, 36.68, 0.02)
    grid = [(round(float(la),4), round(float(lo),4)) for la in lats for lo in lons]
    dem_rows = []
    API = 'https://api.opentopodata.org/v1/srtm30m'
    total_batches = (len(grid) + 99) // 100
    for i in range(0, len(grid), 100):
        batch = grid[i:i+100]
        locs = '|'.join(f'{la},{lo}' for la, lo in batch)
        for attempt in range(3):
            try:
                r = session.get(API, params={'locations': locs}, timeout=30)
                if r.status_code == 200:
                    for res in r.json().get('results', []):
                        if res.get('elevation') is not None:
                            dem_rows.append({
                                'latitude': res['location']['lat'],
                                'longitude': res['location']['lng'],
                                'elevation': res['elevation']
                            })
                    break
                elif r.status_code == 429:
                    time.sleep(2 ** attempt)
            except Exception:
                time.sleep(2 ** attempt)
        time.sleep(1.0)  # Rate limit
        batch_num = i // 100 + 1
        if batch_num % 20 == 0 or batch_num == total_batches:
            print(f'    DEM: {batch_num}/{total_batches} batches ({len(dem_rows)} pts)', end='\r')
    pd.DataFrame(dem_rows).to_csv(dem_path, index=False)
    print(f'\n    DEM done: {len(dem_rows)} points saved')
else:
    print(f'  [skip] DEM already exists ({dem_path.stat().st_size/1e6:.1f} MB)')

print('\n=== All datasets ready ===')


=== Downloading datasets ===
  [downloading] Ookla mobile Q4 2024 (~170 MB) ...
    Done: 195.1 MB
  [querying] OSM telecom sites ...
    [warn] overpass-api.de failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Done: 105 elements (0.0 MB)
  [querying] OSM buildings (major cities) ...
    Done: 30,000 elements (6.3 MB)
  [querying] OSM roads ...
    [warn] overpass-api.de failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Done: 20,000 elements (10.1 MB)
  [querying] Coastline ...
    Done: 242 elements (0.7 MB)
  [computing] DEM grid via OpenTopoData API ...
    DEM: 68/68 batches (6720 pts)
    DEM done: 6720 points saved

=== All datasets ready ===


---
## 1. Data Loading

### Measurement data (for labels)
| Dataset | Source | Records | Measurement |
|---------|--------|---------|-------------|
| **Ookla Speedtest** | Ookla Open Data S3 | ~4,200 Lebanon tiles | Real download/upload/latency |
| **App Measurements** | Project SQLite DB | ~5,400 readings | Real RSRP from device radio |

### Context data (features only — never labels)
| Dataset | Source | Purpose |
|---------|--------|---------|
| **OpenCelliD** | Repo CSV | Tower topology features |
| **OSM Telecom/Buildings/Roads** | Overpass API | Infrastructure density |
| **Coastline** | Overpass API | Distance-to-coast |
| **DEM (SRTM 30m)** | OpenTopoData API | Terrain elevation/slope |


In [5]:
# ──────────────────────────────────────────────────────
# 1a. Load Ookla speed test tiles (PRIMARY LABEL SOURCE)
# ──────────────────────────────────────────────────────
from deadzone_model import load_ookla_tiles

ookla = load_ookla_tiles(DATA_DIR / 'ookla_mobile_q4_2025.parquet', bbox=BBOX)

print(f'Ookla tiles in Lebanon: {len(ookla):,}')
print(f'Download: mean={ookla["avg_d_kbps"].mean()/1000:.1f} Mbps, '
      f'median={ookla["avg_d_kbps"].median()/1000:.1f} Mbps')
print(f'Latency : mean={ookla["avg_lat_ms"].mean():.0f} ms')

# ITU speed tier breakdown
for name, lo, hi in [('Dead zone (<1 Mbps)', 0, 1000),
                      ('Poor (1-5 Mbps)', 1000, 5000),
                      ('Fair (5-25 Mbps)', 5000, 25000),
                      ('Good (>25 Mbps)', 25000, 1e9)]:
    n = ((ookla['avg_d_kbps'] >= lo) & (ookla['avg_d_kbps'] < hi)).sum()
    print(f'  {name:25s}: {n:,} ({n/len(ookla):.1%})')


Ookla tiles in Lebanon: 2,901
Download: mean=31.6 Mbps, median=22.7 Mbps
Latency : mean=33 ms
  Dead zone (<1 Mbps)      : 80 (2.8%)
  Poor (1-5 Mbps)          : 338 (11.7%)
  Fair (5-25 Mbps)         : 1,127 (38.8%)
  Good (>25 Mbps)          : 1,356 (46.7%)


In [6]:
# ──────────────────────────────────────────────────────
# 1b. Load app measurements from SQLite (Tier 1)
# ──────────────────────────────────────────────────────
db_path = SERVER_DIR / 'instance' / 'network_cell_analyzer.db'
app_df = pd.DataFrame()

if db_path.exists():
    conn = sqlite3.connect(str(db_path))
    if 'cell_data' in [r[0] for r in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'").fetchall()]:
        app_df = pd.read_sql(
            'SELECT latitude, longitude, signal_power, snr, operator, '
            'network_type, cell_id, frequency_band FROM cell_data '
            'WHERE latitude IS NOT NULL AND signal_power IS NOT NULL', conn)
    conn.close()

print(f'App measurements: {len(app_df):,} rows')
if len(app_df) > 0:
    print(f'Signal: mean={app_df["signal_power"].mean():.1f} dBm, '
          f'range=[{app_df["signal_power"].min():.0f}, {app_df["signal_power"].max():.0f}]')


App measurements: 0 rows


In [7]:
# ──────────────────────────────────────────────────────
# 1c. Load context data (features only)
# ──────────────────────────────────────────────────────
from deadzone_model import load_opencellid_reference, load_osm_context, load_dem_grid

# OpenCelliD towers
towers = load_opencellid_reference(DATA_DIR / 'opencellid_lebanon_415.csv.gz', bbox=BBOX)
print(f'OpenCelliD towers  : {len(towers):,}')

# OSM telecom
osm_telecom = load_osm_context(DATA_DIR / 'osm_telecom_lebanon.json', bbox=BBOX)
print(f'OSM telecom sites  : {len(osm_telecom):,}')

# OSM buildings
osm_buildings = pd.DataFrame()
bp = DATA_DIR / 'osm_buildings_lebanon.json'
if bp.exists():
    with open(bp) as f:
        braw = json.load(f)
    osm_buildings = pd.DataFrame([
        {'latitude': e['center']['lat'], 'longitude': e['center']['lon']}
        for e in braw.get('elements', []) if 'center' in e
    ])
    if not osm_buildings.empty:
        osm_buildings = osm_buildings[
            osm_buildings['latitude'].between(BBOX[0], BBOX[2]) &
            osm_buildings['longitude'].between(BBOX[1], BBOX[3])
        ].reset_index(drop=True)
print(f'OSM buildings      : {len(osm_buildings):,}')

# OSM roads
osm_roads = pd.DataFrame()
rp = DATA_DIR / 'osm_roads_lebanon.json'
if rp.exists():
    with open(rp) as f:
        rraw = json.load(f)
    osm_roads = pd.DataFrame([
        {'latitude': e['center']['lat'], 'longitude': e['center']['lon']}
        for e in rraw.get('elements', []) if 'center' in e
    ])
    if not osm_roads.empty:
        osm_roads = osm_roads[
            osm_roads['latitude'].between(BBOX[0], BBOX[2]) &
            osm_roads['longitude'].between(BBOX[1], BBOX[3])
        ].reset_index(drop=True)
print(f'OSM roads          : {len(osm_roads):,}')

# Coastline
coastline = pd.DataFrame()
cp = DATA_DIR / 'lebanon_coastline.json'
if cp.exists():
    with open(cp) as f:
        craw = json.load(f)
    pts = []
    for e in craw.get('elements', []):
        for pt in e.get('geometry', []):
            pts.append({'latitude': pt['lat'], 'longitude': pt['lon']})
    coastline = pd.DataFrame(pts)
print(f'Coastline points   : {len(coastline):,}')

# DEM
dem = pd.DataFrame()
for dp in [DATA_DIR / 'lebanon_dem_dense.csv', DATA_DIR / 'lebanon_elevation_grid.csv']:
    if dp.exists():
        dem = load_dem_grid(dp, bbox=BBOX)
        if not dem.empty: break
elev_col = 'elevation' if 'elevation' in dem.columns else (dem.columns[2] if len(dem.columns)>2 else None)
print(f'DEM grid points    : {len(dem):,}')


FileNotFoundError: [Errno 2] No such file or directory: '/content/EECE451-Project/server/data/raw/opencellid_lebanon_415.csv.gz'

---
## 2. Exploratory Data Analysis


In [ ]:
# ──────────────────────────────────────────────────────
# 2a. Geographic coverage and speed distribution
# ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Ookla download speed map
ax = axes[0, 0]
sc = ax.scatter(ookla['longitude'], ookla['latitude'],
               c=np.log10(ookla['avg_d_kbps'].clip(lower=1)),
               s=2, cmap='RdYlGn', alpha=0.5)
plt.colorbar(sc, ax=ax, label='log10(Download kbps)', shrink=0.8)
ax.set_title('Ookla Download Speed (Real Measurements)')
ax.set_ylabel('Latitude')

# Ookla latency map
ax = axes[0, 1]
sc = ax.scatter(ookla['longitude'], ookla['latitude'],
               c=ookla['avg_lat_ms'].clip(upper=200),
               s=2, cmap='RdYlGn_r', alpha=0.5)
plt.colorbar(sc, ax=ax, label='Latency (ms)', shrink=0.8)
ax.set_title('Ookla Latency')

# Tower locations
ax = axes[0, 2]
for op, color in [('Alfa', '#E53935'), ('Touch', '#1E88E5'), ('Unknown', '#999')]:
    sub = towers[towers['operator'] == op]
    if not sub.empty:
        ax.scatter(sub['longitude'], sub['latitude'], s=3, c=color, alpha=0.6,
                  label=f'{op} ({len(sub)})')
ax.set_title('Cell Towers (Feature Context Only)')
ax.legend(markerscale=4, fontsize=8)

# Speed distribution with ITU thresholds
ax = axes[1, 0]
ax.hist(np.log10(ookla['avg_d_kbps'].clip(lower=1)), bins=60,
       color='#42A5F5', edgecolor='white', alpha=0.8)
ax.axvline(x=np.log10(1000), color='red', ls='--', lw=2, label='Dead zone: 1 Mbps (ITU)')
ax.axvline(x=np.log10(5000), color='orange', ls='--', label='Poor: 5 Mbps')
ax.axvline(x=np.log10(25000), color='green', ls='--', label='Good: 25 Mbps')
ax.set_xlabel('log10(Download kbps)')
ax.set_title('Speed Distribution + ITU Thresholds')
ax.legend(fontsize=7)

# Latency distribution
ax = axes[1, 1]
ax.hist(ookla['avg_lat_ms'].clip(upper=300), bins=60, color='#FFA726', edgecolor='white')
ax.axvline(x=100, color='red', ls='--', label='High latency: 100ms')
ax.set_xlabel('Latency (ms)')
ax.set_title('Latency Distribution')
ax.legend(fontsize=8)

# Elevation map
ax = axes[1, 2]
if not dem.empty and elev_col:
    sc = ax.scatter(dem['longitude'], dem['latitude'], c=dem[elev_col],
                   s=2, cmap='terrain', alpha=0.6)
    plt.colorbar(sc, ax=ax, label='Elevation (m)', shrink=0.8)
ax.set_title('Terrain Elevation')

plt.suptitle('EDA — Lebanon Cellular Network', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reports' / 'eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ──────────────────────────────────────────────────────
# 2b. Speed vs Latency — dead zone identification
# ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(np.log10(ookla['avg_d_kbps'].clip(lower=1)),
               ookla['avg_lat_ms'].clip(upper=500),
               c=ookla['tests'].clip(upper=100), s=5, cmap='viridis', alpha=0.5)
plt.colorbar(sc, ax=ax, label='Number of tests', shrink=0.8)
ax.axvline(x=np.log10(1000), color='red', ls='--', lw=1.5)
ax.axhline(y=100, color='red', ls='--', lw=1.5)
ax.fill_between([0, np.log10(1000)], [100, 100], [500, 500],
               alpha=0.1, color='red', label='Dead-zone region')
ax.set_xlabel('log10(Download kbps)')
ax.set_ylabel('Latency (ms)')
ax.set_title('Speed vs Latency — Dead Zone Identification')
ax.legend()
plt.tight_layout()
plt.show()


---
## 3. Dataset Fusion & Labeling

### Academic & Regulatory Justification

| Source | Standard | Threshold |
|--------|----------|----------|
| **3GPP TS 36.133** §9.1.4 | RSRP reporting range | Below **-110 dBm** = near-unusable LTE |
| **Ofcom SRN** (2025) | UK coverage obligation | **2 Mbps at -105 dBm** = minimum "good 4G" |
| **Falkner et al.** (arXiv:1804.05771) | Deployed LTE empirical | RSRP -110→-120 dBm yields **1-5 Mbps** |
| **FCC** (2010 minimum) | US broadband floor | **4/1 Mbps** (lowest-ever; <1 Mbps never broadband) |
| **Apajalahti et al.** (Aalto, 2018) | Crowdsourced LTE correlation | download_kbps ↔ lte_rsrp validated |

**Our threshold: download < 1 Mbps = dead zone** — conservative vs. Ofcom's 2 Mbps.

```
3GPP TS 36.133:  RSRP < -110 dBm  →  near-unusable LTE
arXiv:1804.05771: RSRP -110..-120  →  throughput 1-5 Mbps (empirical)
Ofcom SRN:        RSRP < -105 dBm  →  "not spot" (< 2 Mbps)
FCC (2010):       < 4/1 Mbps       →  not broadband
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Our threshold:    < 1 Mbps         →  dead zone (conservative)
```


In [ ]:
# ──────────────────────────────────────────────────────
# 3a. Build training dataset from real measurements
# ──────────────────────────────────────────────────────
from deadzone_data import build_training_dataset

print('Building training dataset from real measurements ...\n')
dataset = build_training_dataset(
    ookla_df=ookla,
    app_df=app_df if not app_df.empty else None,
    opencellid_df=towers,
    include_topology=True,
)

print(f'\nTotal rows      : {len(dataset):,}')
print(f'Dead-zone rate  : {dataset["is_deadzone"].mean():.1%}')
for src in ['app', 'ookla', 'topology']:
    m = dataset['label_source'] == src
    if m.any():
        print(f'  {src:12s}: {m.sum():>6,} rows, '
              f'DZ rate={dataset.loc[m,"is_deadzone"].mean():.1%}, '
              f'weight={dataset.loc[m,"sample_weight"].mean():.1f}')

measured = dataset['label_source'].isin(['app','ookla']).sum()
print(f'\nReal measurements: {measured:,}/{len(dataset):,} ({measured/len(dataset):.0%})')


In [ ]:
# ──────────────────────────────────────────────────────
# 3b. Compute COST-231 RSRP for topology gap-filler rows
# ──────────────────────────────────────────────────────
from sklearn.neighbors import BallTree
from deadzone_propagation import compute_propagation_features

needs_rsrp = (
    (dataset['label_source'] == 'topology') &
    (~dataset.get('cost231_predicted_rsrp', pd.Series(dtype=float)).notna()
     if 'cost231_predicted_rsrp' in dataset.columns
     else pd.Series(True, index=dataset.index))
)

if needs_rsrp.any() and len(towers) > 0:
    print(f'Computing COST-231 for {needs_rsrp.sum()} topology rows ...')
    ref_tree = BallTree(
        np.radians(towers[['latitude','longitude']].values.astype(float)),
        metric='haversine')
    rows_n = dataset[needs_rsrp]
    q = np.radians(rows_n[['latitude','longitude']].values.astype(float))
    _, idxs = ref_tree.query(q, k=1)
    vals = []
    for i in range(len(rows_n)):
        nr = towers.iloc[idxs[i,0]]
        pf = compute_propagation_features(
            float(rows_n.iloc[i]['latitude']), float(rows_n.iloc[i]['longitude']),
            str(rows_n.iloc[i].get('operator','')),
            str(rows_n.iloc[i].get('network_type','4G')),
            rows_n.iloc[i].get('frequency_band'),
            float(nr['latitude']), float(nr['longitude']))
        vals.append(pf['cost231_predicted_rsrp'])
    dataset.loc[needs_rsrp, 'cost231_predicted_rsrp'] = vals
    print('Done.')
else:
    print('No topology rows need RSRP computation')


In [ ]:
# ──────────────────────────────────────────────────────
# 3c. Visualize labeling results
# ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Class balance
ax = axes[0]
cts = dataset['is_deadzone'].value_counts().sort_index()
bars = ax.bar(['Not Dead-Zone', 'Dead-Zone'], cts.values, color=['#66BB6A', '#EF5350'])
for b, v in zip(bars, cts.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+20, f'{v:,}', ha='center', fontweight='bold')
ax.set_title('Class Balance'); ax.set_ylabel('Count')

# By source tier
ax = axes[1]
src_dz = dataset.groupby('label_source')['is_deadzone'].agg(['sum','count'])
src_dz['not_dz'] = src_dz['count'] - src_dz['sum']
src_dz[['not_dz','sum']].plot(kind='bar', stacked=True, ax=ax,
    color=['#66BB6A','#EF5350'], legend=False)
ax.set_title('Labels by Source Tier'); ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(['OK','Dead-Zone'], fontsize=9)

# Map
ax = axes[2]
ok = dataset[dataset['is_deadzone']==0]
dz = dataset[dataset['is_deadzone']==1]
ax.scatter(ok['longitude'], ok['latitude'], s=2, c='#66BB6A', alpha=0.3, label=f'OK ({len(ok):,})')
ax.scatter(dz['longitude'], dz['latitude'], s=8, c='#EF5350', alpha=0.8, label=f'DZ ({len(dz):,})')
ax.legend(markerscale=3)
ax.set_title('Dead-Zone Map'); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()


---
## 4. Feature Engineering (43 Features)

| Category | # | Key features |
|----------|---|-------------|
| Spatial | 3 | lat, lon, distance_to_coast_km |
| Tower Topology | 10 | density 1km/3km, nearest distance, mean signal/range |
| Propagation | 6 | COST-231 path loss, predicted RSRP, excess loss |
| Terrain | 5 | elevation, slope, relief, LOS obstruction |
| Ookla Performance | 6 | IDW-interpolated speed/latency |
| OSM Urban | 4 | building/road/telecom density, urban class |
| App Aggregates | 6 | H3 mean signal, std, deadzone fraction |
| Categorical | 5 | operator, network_type, h3_res7/9, urban_class |


In [ ]:
# ──────────────────────────────────────────────────────
# 4a. Build FeatureContext
# ──────────────────────────────────────────────────────
from deadzone_features import (
    FeatureContext, build_feature_dataframe,
    NUMERIC_FEATURES_V3, CATEGORICAL_FEATURES_V3
)

if not dem.empty and elev_col:
    dem_lats, dem_lons, dem_elevs = dem['latitude'].values, dem['longitude'].values, dem[elev_col].values
else:
    dem_lats = dem_lons = dem_elevs = None

ctx = FeatureContext(
    ref_cells=towers,
    ookla_df=ookla if not ookla.empty else None,
    dem_lats=dem_lats, dem_lons=dem_lons, dem_elevations=dem_elevs,
    osm_telecom_df=osm_telecom if not osm_telecom.empty else None,
    osm_buildings_df=osm_buildings if not osm_buildings.empty else None,
    osm_roads_df=osm_roads if not osm_roads.empty else None,
    coast_df=coastline if not coastline.empty else None,
)
print(f'Features: {len(NUMERIC_FEATURES_V3)} numeric + {len(CATEGORICAL_FEATURES_V3)} cat')


In [ ]:
%%time
# ──────────────────────────────────────────────────────
# 4b. Compute features for all training rows
# ──────────────────────────────────────────────────────
feature_df = build_feature_dataframe(dataset, ctx)
print(f'Feature matrix: {feature_df.shape[0]:,} x {feature_df.shape[1]}')

completeness = (1 - feature_df[NUMERIC_FEATURES_V3].isnull().mean()) * 100
print(f'\nFeature completeness:')
for f, pct in completeness.sort_values().items():
    print(f'  {f:.<42} {pct:5.1f}%')


In [ ]:
# ──────────────────────────────────────────────────────
# 4c. Feature correlation with dead-zone label
# ──────────────────────────────────────────────────────
corr = feature_df[NUMERIC_FEATURES_V3].corrwith(
    dataset['is_deadzone'].astype(float)).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 12))
colors = ['#EF5350' if v > 0 else '#42A5F5' for v in corr.values]
ax.barh(range(len(corr)), corr.values, color=colors)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.index, fontsize=9)
ax.set_xlabel('Correlation with Dead-Zone')
ax.set_title('Feature Correlation Analysis')
ax.axvline(x=0, color='black', lw=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


---
## 5. Model Training
Dual LightGBM with spatial CV (H3 resolution 5 hexagons).


In [ ]:
# ──────────────────────────────────────────────────────
# 5a. Prepare labels
# ──────────────────────────────────────────────────────
from deadzone_training import train_dual_model

TUNE = False  # Set True for Optuna hyperparameter search

labels = dataset[['is_deadzone','signal_target','sample_weight',
                   'regression_weight','label_source']].copy()

print(f'Training: {len(labels):,} samples, DZ rate: {labels["is_deadzone"].mean():.1%}')
for src in labels['label_source'].unique():
    m = labels['label_source'] == src
    print(f'  {src}: n={m.sum():,}, weight={labels.loc[m,"sample_weight"].mean():.1f}')


In [ ]:
%%time
# ──────────────────────────────────────────────────────
# 5b. Train dual LightGBM
# ──────────────────────────────────────────────────────
result = train_dual_model(feature_df=feature_df, labels=labels, tune=TUNE)

regressor = result['regressor']
classifier = result['classifier']
metrics = result['metrics']
reg_params = result['reg_params']
cls_params = result['cls_params']
print('Training complete.')


---
## 6. Evaluation


In [ ]:
# ──────────────────────────────────────────────────────
# 6a. Metrics
# ──────────────────────────────────────────────────────
print('=' * 55)
print('MODEL EVALUATION')
print('=' * 55)
if 'regressor' in metrics:
    print('\n--- Signal Regressor ---')
    for k in ['rmse','mae','r2','median_ae']:
        if k in metrics['regressor']: print(f'  {k:15s}: {metrics["regressor"][k]:.3f}')
if 'classifier' in metrics:
    print('\n--- Dead-Zone Classifier ---')
    for k in ['accuracy','roc_auc','pr_auc','precision','recall','f1']:
        if k in metrics['classifier']: print(f'  {k:15s}: {metrics["classifier"][k]:.4f}')
if 'per_tier' in metrics:
    print('\n--- Per-Tier ---')
    for t, tm in metrics['per_tier'].items():
        print(f'  {t}: PR-AUC={tm.get("pr_auc",0):.3f}')
if 'spatial_cv' in metrics:
    scv = metrics['spatial_cv']
    print(f'\n--- Spatial CV ---')
    print(f'  PR-AUC: {scv.get("mean_pr_auc",0):.4f} +/- {scv.get("std_pr_auc",0):.4f}')
print('=' * 55)


In [ ]:
# ──────────────────────────────────────────────────────
# 6b. Prediction distribution plots
# ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Actual vs predicted (from OOF regressor predictions)
oof_preds = result['oof_predictions']
ax = axes[0]
vr = ~np.isnan(labels['signal_target'].values)
if vr.any():
    ax.scatter(labels['signal_target'].values[vr], oof_preds[vr],
              s=3, alpha=0.3, c='#42A5F5')
    lims = [labels['signal_target'].dropna().min(), labels['signal_target'].dropna().max()]
    ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('Actual (dBm)'); ax.set_ylabel('Predicted (dBm)')
ax.set_title('Regressor: Actual vs Predicted')

# Classifier probability distribution
# Build the feature matrix the classifier expects (with signal_pred_oof)
cls_feature_df = feature_df.copy()
cls_feature_df['signal_pred_oof'] = oof_preds
ax = axes[1]
proba = classifier.predict_proba(cls_feature_df)[:, 1]
y = labels['is_deadzone'].values
ax.hist(proba[y==0], bins=50, alpha=0.6, color='#66BB6A', label='OK', density=True)
ax.hist(proba[y==1], bins=50, alpha=0.6, color='#EF5350', label='DZ', density=True)
ax.axvline(x=0.5, color='black', ls='--')
ax.set_title('Classifier Probabilities'); ax.legend()

# Residuals
ax = axes[2]
if vr.any():
    res = labels['signal_target'].values[vr] - oof_preds[vr]
    ax.hist(res, bins=50, color='#AB47BC', edgecolor='white')
    ax.set_title(f'Residuals (MAE={np.abs(res).mean():.1f} dBm)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reports' / 'eval_plots.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 7. SHAP Explainability


In [ ]:
# ──────────────────────────────────────────────────────
# 7a. SHAP values + importance plot
# ──────────────────────────────────────────────────────
import shap

lgb_clf = classifier.named_steps['cls']
prep = classifier.named_steps['pre']
n_sample = min(500, len(cls_feature_df))
idx_s = np.random.RandomState(42).choice(len(cls_feature_df), n_sample, replace=False)
X_s = prep.transform(cls_feature_df.iloc[idx_s])

explainer = shap.TreeExplainer(lgb_clf)
sv = explainer.shap_values(X_s)
if isinstance(sv, list): sv = sv[1]

# Build feature name list matching preprocessor output
fnames = NUMERIC_FEATURES_V3 + CATEGORICAL_FEATURES_V3 + ['signal_pred_oof']
if sv.shape[1] != len(fnames):
    fnames = [f'f{i}' for i in range(sv.shape[1])]

mas = np.abs(sv).mean(axis=0)
si = np.argsort(mas)[::-1]
top = min(20, len(si))

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(range(top), mas[si[:top]][::-1], color='#42A5F5')
ax.set_yticks(range(top))
ax.set_yticklabels([fnames[i] for i in si[:top]][::-1], fontsize=9)
ax.set_xlabel('Mean |SHAP|')
ax.set_title('Top 20 Feature Importances (SHAP)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reports' / 'shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10:')
for i in range(min(10, len(si))):
    print(f'  {i+1}. {fnames[si[i]]}: {mas[si[i]]:.4f}')


In [ ]:
# ──────────────────────────────────────────────────────
# 7b. Example predictions with reasons
# ──────────────────────────────────────────────────────
from deadzone_explain import compute_shap_reasons

examples = []
for val in [1, 0]:
    m = dataset['is_deadzone'] == val
    if m.any(): examples.extend(dataset[m].index[:3].tolist())

# Feature names matching the preprocessor output
reason_fnames = NUMERIC_FEATURES_V3 + CATEGORICAL_FEATURES_V3 + ['signal_pred_oof']

print('Example Predictions')
print('=' * 65)
for ex_idx in examples:
    pos = dataset.index.get_loc(ex_idx)
    row = cls_feature_df.iloc[pos:pos+1]
    p = classifier.predict_proba(row)[0, 1]
    lat, lon = dataset.loc[ex_idx, 'latitude'], dataset.loc[ex_idx, 'longitude']
    actual = 'DZ' if dataset.loc[ex_idx, 'is_deadzone'] else 'OK'
    lsrc = dataset.loc[ex_idx, 'label_source']
    print(f'\n({lat:.4f}, {lon:.4f}) [{lsrc}] actual={actual} pred={p:.3f}')
    try:
        row_transformed = prep.transform(row)
        reasons = compute_shap_reasons(classifier, row_transformed, reason_fnames, top_k=3)
        for ri, r in enumerate(reasons, 1):
            print(f'  {ri}. {r}')
    except Exception as e:
        print(f'  (reasons unavailable: {e})')
print('\n' + '=' * 65)


---
## 8. Export Model Bundle


In [ ]:
# ──────────────────────────────────────────────────────
# 8a. Save model bundle
# ──────────────────────────────────────────────────────
import shutil
from deadzone_model import MODEL_VERSION, reference_subset_for_runtime

bundle = {
    'model_version': MODEL_VERSION,
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'model': classifier, 'regressor': regressor,
    'reg_params': reg_params, 'cls_params': cls_params,
    'metrics': metrics,
    'metadata': {
        'training_row_count': len(dataset),
        'tier_counts': dataset['label_source'].value_counts().to_dict(),
        'deadzone_rate': float(dataset['is_deadzone'].mean()),
        'measured_label_pct': float(dataset['label_source'].isin(['app','ookla']).mean()),
        'bbox': BBOX,
    },
    'reference_cells': reference_subset_for_runtime(towers),
    'ookla_tiles': ookla, 'osm_context': osm_telecom,
    'dem_grid': dem, 'osm_buildings': osm_buildings,
    'osm_roads': osm_roads, 'coastline': coastline,
}

model_path = OUTPUT_DIR / 'deadzone_model.joblib'
joblib.dump(bundle, model_path, compress=3)
print(f'Model: {model_path} ({model_path.stat().st_size/1e6:.1f} MB)')

# ── Copy to Google Drive ──
drive_model = DRIVE_OUT / 'deadzone_model.joblib'
shutil.copy2(model_path, drive_model)
print(f'Copied to Drive: {drive_model}')


In [ ]:
# ──────────────────────────────────────────────────────
# 8b. Save report + dataset
# ──────────────────────────────────────────────────────
report = {
    'trained_at': bundle['trained_at'], 'model_version': MODEL_VERSION,
    'training_rows': len(dataset),
    'deadzone_rate': float(dataset['is_deadzone'].mean()),
    'measured_label_pct': float(dataset['label_source'].isin(['app','ookla']).mean()),
    'metrics': metrics, 'reg_params': reg_params, 'cls_params': cls_params,
    'tier_counts': dataset['label_source'].value_counts().to_dict(),
}
with open(OUTPUT_DIR / 'reports' / 'summary.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

dataset.to_csv(OUTPUT_DIR / 'prepared_deadzone_dataset.csv', index=False)
print('Report + dataset saved.')

# ── Copy everything to Google Drive ──
import shutil
for fname in ['summary.json']:
    src = OUTPUT_DIR / 'reports' / fname
    if src.exists(): shutil.copy2(src, DRIVE_OUT / fname)
for fname in ['eda_overview.png', 'eval_plots.png', 'shap_importance.png']:
    src = OUTPUT_DIR / 'reports' / fname
    if src.exists(): shutil.copy2(src, DRIVE_OUT / fname)
shutil.copy2(OUTPUT_DIR / 'prepared_deadzone_dataset.csv',
             DRIVE_OUT / 'prepared_deadzone_dataset.csv')
print(f'All outputs copied to {DRIVE_OUT}')


In [ ]:
# ──────────────────────────────────────────────────────
# 8c. Verify inference works
# ──────────────────────────────────────────────────────
from deadzone_model import predict_deadzone

pred = predict_deadzone(
    model_path=str(model_path),
    latitude=33.8938, longitude=35.5018,
    operator='Alfa', network_type='4G',
)

print('Test: Central Beirut (33.8938, 35.5018)')
if pred:
    for k, v in pred.items():
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
    print('\nModel ready for deployment.')


In [ ]:
# ──────────────────────────────────────────────────────
# 8d. List saved files on Google Drive
# ──────────────────────────────────────────────────────
print('\nFiles saved to Google Drive:')
print('=' * 50)
for f in sorted(DRIVE_OUT.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:45s} {size_mb:7.2f} MB')
print(f'\nDrive path: {DRIVE_OUT}')
print('You can download these from Google Drive or use them directly.')

---
## Summary

| Aspect | Detail |
|--------|--------|
| **Models** | LightGBM Regressor (Huber) + Classifier (binary, stacked) |
| **Features** | 43 (spatial, topology, propagation, terrain, Ookla, OSM, app) |
| **Labels** | Real measurements: Ookla speed (ITU-T Y.1541) + app signal (3GPP TS 36.133) |
| **Validation** | Spatial CV with H3 hexagons (no geographic leakage) |
| **Explainability** | SHAP TreeExplainer with human-readable reasons |
| **Thresholds** | Speed < 1 Mbps (Falkner et al., arXiv:1804.05771) OR RSRP < -110 dBm |

### References
1. 3GPP TS 36.133, §9.1.4 — RSRP measurement reporting range
2. Falkner et al., *Performance Evaluation of a Deployed 4G LTE Network*, arXiv:1804.05771
3. Ofcom, *Shared Rural Network Coverage Obligations* (2025) — 2 Mbps / -105 dBm
4. FCC Broadband Speed Benchmark (2010–2024)
5. Apajalahti et al., *Correlation-Based Feature Mapping of Crowdsourced LTE Data*, Aalto (2018)
